# 03. Product-Level Reorder Frequency & Restock Prediction

**Lead Responsible**: Peiris W.S.V (`IT24103792`)  
**Module**: IT3091 – Machine Learning | Group 2026-AI-15  
**Track**: Industry Explorer Track  

---

## Objectives
1. Engineer product-level features from transaction histories (reorder count, repurchase lag from `days_since_prior_order`, department velocities).
2. Train baseline **Logistic Regression** classifier to predict high-frequency reorder items.
3. Benchmark non-linear classification models: **Decision Tree** and **Random Forest**.
4. Evaluate performance using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [2]:
import sys
import time
from pathlib import Path

# Resolve project root dynamically
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn Modeling, Tuning & Validation Modules
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_val_score, 
    RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    HistGradientBoostingClassifier
)

# Project modular utilities from src/
from src.utils import setup_plot_style, save_figure, export_summary_table

setup_plot_style()
print(f"[OK] Python Interpreter : {sys.executable}")
print(f"[OK] Project Root       : {PROJECT_ROOT}")


Matplotlib is building the font cache; this may take a moment.


[OK] Python Interpreter : /Users/supunvidarshana/Desktop/My projects/supermarket-basket-reorder-analytics/.venv/bin/python3.13
[OK] Project Root       : /Users/supunvidarshana/Desktop/My projects/supermarket-basket-reorder-analytics


---
## 1. Data Ingestion: Train and Test Partitions
We ingest `train_transactions.parquet` and `test_transactions.parquet` generated by the preprocessing pipeline.


In [3]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = DATA_DIR / "train_transactions.parquet"
TEST_PATH = DATA_DIR / "test_transactions.parquet"

assert TRAIN_PATH.exists(), f"Missing: {TRAIN_PATH}. Check data/processed/"
assert TEST_PATH.exists(), f"Missing: {TEST_PATH}. Check data/processed/"

print(f"Loading partitions from {DATA_DIR}...")
t0 = time.time()
df_train = pd.read_parquet(TRAIN_PATH)
df_test = pd.read_parquet(TEST_PATH)

print(f"✅ Loaded Train Partition : {len(df_train):,} rows ({time.time() - t0:.2f}s)")
print(f"✅ Loaded Test Partition  : {len(df_test):,} rows")


Loading partitions from /Users/supunvidarshana/Desktop/My projects/supermarket-basket-reorder-analytics/data/processed...
✅ Loaded Train Partition : 1,615,600 rows (3.65s)
✅ Loaded Test Partition  : 403,901 rows


---
## 2. Leakage-Free Feature Engineering
To prevent **target and data leakage**, all aggregated statistical features (such as historical product reorder rates and user repurchase frequencies) must be computed **strictly on the training dataset** and mapped onto the test dataset.


In [4]:
print("Constructing Supervised Feature Matrices...")

# 1. Product-Level Aggregations 
prod_stats = df_train.groupby("product_id", observed=False).agg(
    prod_total_orders=("order_id", "count"),
    prod_reorder_rate=("reordered", "mean")
).reset_index()

# 2. User-Level Aggregations 
user_stats = df_train.groupby("user_id", observed=False).agg(
    user_total_orders=("order_number", "max"),
    user_reorder_rate=("reordered", "mean"),
    user_avg_basket_size=("basket_size", "mean") if "basket_size" in df_train.columns else ("order_number", "count")
).reset_index()

# 3. Merge aggregates onto df_train and df_test
def merge_and_prepare_features(df_source, p_stats, u_stats):
    df_merged = df_source.merge(p_stats, on="product_id", how="left")
    df_merged = df_merged.merge(u_stats, on="user_id", how="left")
    return df_merged

df_train_feat = merge_and_prepare_features(df_train, prod_stats, user_stats)
df_test_feat = merge_and_prepare_features(df_test, prod_stats, user_stats)

# Select relevant feature columns
feature_cols = [
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order_scaled",
    "add_to_cart_order",
    "basket_size_capped",
    "is_first_order",
    "prod_total_orders",
    "prod_reorder_rate",
    "user_total_orders",
    "user_reorder_rate",
    "user_avg_basket_size"
]
target_col = "reordered"

# Ensure all feature columns exist
available_cols = [c for c in feature_cols if c in df_train_feat.columns]

# Sample training set if needed for rapid iteration, while retaining a robust sample
TRAIN_SAMPLE_SIZE = min(150000, len(df_train_feat))
TEST_SAMPLE_SIZE = min(30000, len(df_test_feat))

df_train_sample = df_train_feat.sample(n=TRAIN_SAMPLE_SIZE, random_state=42)
df_test_sample = df_test_feat.sample(n=TEST_SAMPLE_SIZE, random_state=42)

X_train = df_train_sample[available_cols].fillna(0).copy()
y_train = df_train_sample[target_col].copy()

X_test = df_test_sample[available_cols].fillna(0).copy()
y_test = df_test_sample[target_col].copy()

print(f"✅ Training Matrix : {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"✅ Test Matrix     : {X_test.shape[0]:,} rows x {X_test.shape[1]} features")
print(f"Train Reorder Rate : {y_train.mean()*100:.2f}% | Test Reorder Rate: {y_test.mean()*100:.2f}%")


Constructing Supervised Feature Matrices...
✅ Training Matrix : 150,000 rows x 11 features
✅ Test Matrix     : 30,000 rows x 11 features
Train Reorder Rate : 59.02% | Test Reorder Rate: 58.94%


---
## 3. Model 1: Logistic Regression Baseline
We train an $L_2$-regularized Logistic Regression model as the linear benchmark to measure baseline predictive power and compute odds ratios.


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

print("Tuning Logistic Regression across regularization varieties (C) with StandardScaler...")

# 1. Define Pipeline with StandardScaler to ensure rapid convergence
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000, 
        random_state=42, 
        class_weight="balanced", 
        solver="lbfgs"
    ))
])

# Parameter names in a Pipeline use the '<step_name>__<parameter>' format
lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0]
}

lr_search = GridSearchCV(
    estimator=lr_pipe,
    param_grid=lr_param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)
lr_search.fit(X_train, y_train)

# 2. Extract best pipeline model
lr_model = lr_search.best_estimator_
best_c = lr_search.best_params_["classifier__C"]
print(f"✅ Best Logistic Regression Variety: C = {best_c} (CV ROC-AUC: {lr_search.best_score_:.4f})")

# 3. Extract Odds Ratios from the underlying classifier step
fitted_classifier = lr_model.named_steps["classifier"]
odds_df = pd.DataFrame({
    "Feature": available_cols,
    "Coefficient": fitted_classifier.coef_[0],
    "Odds_Ratio": np.exp(fitted_classifier.coef_[0])
}).sort_values(by="Coefficient", ascending=False)

print("\nTop 5 Features by Odds Ratio:")
display(odds_df.head(5))


Tuning Logistic Regression across regularization varieties (C) with StandardScaler...
✅ Best Logistic Regression Variety: C = 1.0 (CV ROC-AUC: 0.8579)

Top 5 Features by Odds Ratio:


,Feature,Coefficient,Odds_Ratio
9,user_reorder_rate,1.366848,3.922968
7,prod_reorder_rate,0.508454,1.662719
0,order_number,0.449293,1.567204
10,user_avg_basket_size,0.322967,1.381220
4,basket_size_capped,0.158967,1.172299


---
## 4. Model 2: Decision Tree & Capacity Experiments (Bias-Variance Check)
We analyze 3 depth variations to demonstrate capacity constraints for the viva defense:
* **Small Capacity (max_depth=4)**: High Bias / Underfitting risk
* **Medium Capacity (max_depth=10)**: Balanced regularized tree
* **Large Capacity (max_depth=22)**: High Variance / Overfitting risk


In [7]:
print("Tuning Decision Tree across capacity and depth varieties...")
dt_param_grid = {
    "max_depth": [4, 8, 12, 16],
    "min_samples_leaf": [10, 25, 50]
}

dt_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=dt_param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)
dt_search.fit(X_train, y_train)

# Select best variety
best_dt = dt_search.best_estimator_
print(f"✅ Best Decision Tree Variety: {dt_search.best_params_} (CV ROC-AUC: {dt_search.best_score_:.4f})")


Tuning Decision Tree across capacity and depth varieties...
✅ Best Decision Tree Variety: {'max_depth': 12, 'min_samples_leaf': 50} (CV ROC-AUC: 0.8494)
